# Pruning

In pruning, we remove the weights (or whole neurons \ filters) based on calculated importance. The importance can be defined in several ways, the simple case is by using the $l_n$-norm.

In [ ]:
import torch
from torchvision.models import resnet18

model = resnet18(pretrained=True).eval()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
import torch.nn.utils.prune as prune

In [ ]:
module = model.conv1
module

Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)

In [ ]:
module.weight.shape

torch.Size([64, 3, 7, 7])

Let's try $l_1$-pruning and remove $5\%$ of weights with the lowest energy.

In [ ]:
prune.l1_unstructured(module, name="weight", amount=0.05) # zero-out 5%

Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)

In [ ]:
print(list(module.named_parameters()))

[('weight_orig', Parameter containing:
tensor([[[[-1.0419e-02, -6.1356e-03, -1.8098e-03,  ...,  5.6615e-02,
            1.7083e-02, -1.2694e-02],
          [ 1.1083e-02,  9.5276e-03, -1.0993e-01,  ..., -2.7124e-01,
           -1.2907e-01,  3.7424e-03],
          [-6.9434e-03,  5.9089e-02,  2.9548e-01,  ...,  5.1972e-01,
            2.5632e-01,  6.3573e-02],
          ...,
          [-2.7535e-02,  1.6045e-02,  7.2595e-02,  ..., -3.3285e-01,
           -4.2058e-01, -2.5781e-01],
          [ 3.0613e-02,  4.0960e-02,  6.2850e-02,  ...,  4.1384e-01,
            3.9359e-01,  1.6606e-01],
          [-1.3736e-02, -3.6746e-03, -2.4084e-02,  ..., -1.5070e-01,
           -8.2230e-02, -5.7828e-03]],

         [[-1.1397e-02, -2.6619e-02, -3.4641e-02,  ...,  3.2521e-02,
            6.6221e-04, -2.5743e-02],
          [ 4.5687e-02,  3.3603e-02, -1.0453e-01,  ..., -3.1253e-01,
           -1.6051e-01, -1.2826e-03],
          [-8.3730e-04,  9.8420e-02,  4.0210e-01,  ...,  7.0789e-01,
            3.6887e

In [ ]:
print(list(module.named_buffers()))

[('weight_mask', tensor([[[[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]],

         [[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]],

         [[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]]],


        [[[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 

As you see, PyTorch's pruning replaces `weight` with `weight_orig` that stores original weights and `weight_mask` that says which weights are actually being used. During forward, PyTorch manually set the `weight` by applying mask to `weight_orig`

In [ ]:
module._forward_pre_hooks

OrderedDict([(1, <torch.nn.utils.prune.L1Unstructured at 0x7b478adb6630>)])

To make the pruning permanent, we need to call `prune.remove`.

In [ ]:
prune.remove(module, 'weight')

Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)

In [ ]:
print(list(module.named_parameters()))

[('weight', Parameter containing:
tensor([[[[-1.0419e-02, -6.1356e-03, -1.8098e-03,  ...,  5.6615e-02,
            1.7083e-02, -1.2694e-02],
          [ 1.1083e-02,  9.5276e-03, -1.0993e-01,  ..., -2.7124e-01,
           -1.2907e-01,  3.7424e-03],
          [-6.9434e-03,  5.9089e-02,  2.9548e-01,  ...,  5.1972e-01,
            2.5632e-01,  6.3573e-02],
          ...,
          [-2.7535e-02,  1.6045e-02,  7.2595e-02,  ..., -3.3285e-01,
           -4.2058e-01, -2.5781e-01],
          [ 3.0613e-02,  4.0960e-02,  6.2850e-02,  ...,  4.1384e-01,
            3.9359e-01,  1.6606e-01],
          [-1.3736e-02, -3.6746e-03, -2.4084e-02,  ..., -1.5070e-01,
           -8.2230e-02, -5.7828e-03]],

         [[-1.1397e-02, -2.6619e-02, -3.4641e-02,  ...,  3.2521e-02,
            6.6221e-04, -2.5743e-02],
          [ 4.5687e-02,  3.3603e-02, -1.0453e-01,  ..., -3.1253e-01,
           -1.6051e-01, -1.2826e-03],
          [-8.3730e-04,  9.8420e-02,  4.0210e-01,  ...,  7.0789e-01,
            3.6887e-01, 

In [ ]:
print(list(module.named_buffers()))

[]


Let's now try structured pruning and remove some of the output channels.

In [ ]:
# weight.shape == (out_channels, in_channels, kernel_h, kernel_w)
prune.ln_structured(module, name="weight", amount=0.10, n=1, dim=0) # zero-out 10%

Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)

In [ ]:
print(list(module.named_buffers()))

[('weight_mask', tensor([[[[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]],

         [[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]],

         [[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]]],


        [[[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 

Even though the weights are zeroed, the module itself still shows 64 output channels. Even after making pruning permanent

In [ ]:
prune.remove(module, 'weight')

Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)

We can try to manually re-create the layer and transfer the weights

In [ ]:
import torch
import torch.nn as nn

def get_nonzero_out_channels(conv: torch.nn.Conv2d) -> torch.Tensor:
    """
    Returns indices of Conv2d output channels whose weights are not all zero.

    Conv2d weight shape:
        [out_channels, in_channels / groups, kernel_h, kernel_w]
    """
    with torch.no_grad():
        # True for output channels that contain at least one nonzero weight
        keep_mask = conv.weight.detach().abs().sum(dim=(1, 2, 3)) != 0

        keep_out_channels = torch.where(keep_mask)[0]

    return keep_out_channels


def shrink_conv2d_out_channels(
    conv: nn.Conv2d,
    keep_out_channels: torch.Tensor | list[int],
) -> nn.Conv2d:
    """
    Create a new Conv2d with fewer output channels.

    Original weight shape:
        [out_channels, in_channels / groups, kh, kw]

    This function keeps selected output channels along dim=0.
    """

    if isinstance(keep_out_channels, list):
        keep_out_channels = torch.tensor(keep_out_channels, dtype=torch.long)

    keep_out_channels = keep_out_channels.to(conv.weight.device)

    old_out_channels = conv.out_channels
    new_out_channels = len(keep_out_channels)

    assert new_out_channels > 0
    assert keep_out_channels.min() >= 0
    assert keep_out_channels.max() < old_out_channels

    # This simple version supports normal Conv2d.
    # Grouped/depthwise conv needs extra care.
    assert conv.groups == 1, "This helper assumes groups=1."

    new_conv = nn.Conv2d(
        in_channels=conv.in_channels,
        out_channels=new_out_channels,
        kernel_size=conv.kernel_size,
        stride=conv.stride,
        padding=conv.padding,
        dilation=conv.dilation,
        groups=conv.groups,
        bias=conv.bias is not None,
        padding_mode=conv.padding_mode,
        device=conv.weight.device,
        dtype=conv.weight.dtype,
    )

    with torch.no_grad():
        new_conv.weight.copy_(conv.weight[keep_out_channels, :, :, :])

        if conv.bias is not None:
            new_conv.bias.copy_(conv.bias[keep_out_channels])

    return new_conv

In [ ]:
keep_out_channels = get_nonzero_out_channels(module)
keep_out_channels

tensor([ 0,  1,  3,  5,  6,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20,
        21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 37, 39, 40,
        41, 42, 43, 44, 45, 46, 47, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59,
        60, 61, 62, 63])

In [ ]:
new_module = shrink_conv2d_out_channels(module, keep_out_channels)

In [ ]:
new_module

Conv2d(3, 58, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)

Nice, we got a smaller Conv2d. Now, we can replace it in the original ResNet.

In [ ]:
model.conv1 = new_module

In [ ]:
model

ResNet(
  (conv1): Conv2d(3, 58, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [ ]:
model(torch.randn((1, 3, 224, 224)))

RuntimeError: running_mean should contain 58 elements not 64

Issue: different modules are dependent on each other. If we removed some channels in Conv2d, we need to remove input channels in subsequent modules. While for sequential models it is not too hard to do yourself, it gets very complicated for advanced layer dependencies.

This is where `Torch-Pruning` (https://github.com/VainF/Torch-Pruning) can help.

In [ ]:
!pip install torch-pruning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 2.0 MB/s eta 0:00:00


In [ ]:
import torch_pruning as tp

In [ ]:
model = resnet18(pretrained=True).eval()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
module = model.conv1
prune.ln_structured(module, name="weight", amount=0.10, n=1, dim=0) # zero-out 10%

Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)

In [ ]:
prune.remove(module, 'weight')

Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)

In [ ]:
def get_prune_idx(conv: torch.nn.Conv2d) -> torch.Tensor:
    """
    Returns indices of Conv2d output channels whose weights are not all zero.

    Conv2d weight shape:
        [out_channels, in_channels / groups, kernel_h, kernel_w]
    """
    with torch.no_grad():
        # True for output channels that contain at least one nonzero weight
        prune_mask = conv.weight.detach().abs().sum(dim=(1, 2, 3)) == 0

        prune_idx = torch.where(prune_mask)[0]

    return prune_idx

In [ ]:
prune_idx = get_prune_idx(module)
prune_idx

tensor([ 2,  4,  7, 36, 38, 48])

In [ ]:
# 1. Build dependency graph for a resnet18. This requires a dummy input for forwarding
DG = tp.DependencyGraph().build_dependency(model, example_inputs=torch.randn(1,3,224,224))

# 2. To prune the output channels of model.conv1, we need to find the corresponding group with a pruning function and pruning indices.
group = DG.get_pruning_group(model.conv1, tp.prune_conv_out_channels, idxs=prune_idx )

# 3. Do the pruning
if DG.check_pruning_group(group): # avoid over-pruning, i.e., channels=0.
    group.prune()


We pass the indexes of the channels that we wanted to prune and the library understands all the dependencies and automatically re-defines all the layers involved.

In [ ]:
model

ResNet(
  (conv1): Conv2d(3, 58, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(58, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(58, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 58, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(58, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(58, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

We can see what dependencies the library have found:

In [ ]:
print(group.details()) # or print(group)


--------------------------------
          Pruning Group
--------------------------------
[0] prune_out_channels on conv1 (Conv2d(3, 58, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)) => prune_out_channels on conv1 (Conv2d(3, 58, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)), idxs (6) =[tensor(2), tensor(4), tensor(7), tensor(36), tensor(38), tensor(48)]  (Pruning Root)
[1] prune_out_channels on conv1 (Conv2d(3, 58, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)) => prune_out_channels on bn1 (BatchNorm2d(58, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)), idxs (6) =[tensor(2), tensor(4), tensor(7), tensor(36), tensor(38), tensor(48)] 
[2] prune_out_channels on bn1 (BatchNorm2d(58, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)) => prune_out_channels on _ElementWiseOp_20(ReluBackward0), idxs (6) =[tensor(2), tensor(4), tensor(7), tensor(36), tensor(38), tensor(48)] 
[3] prune_out_channels on _ElementW

You can do pruning via the lib as well

In [ ]:
import torch
from torchvision.models import resnet18
import torch_pruning as tp

model = resnet18(pretrained=True)
example_inputs = torch.randn(1, 3, 224, 224)

# 1. Importance criterion, here we calculate the L2 Norm of grouped weights as the importance score
imp = tp.importance.GroupMagnitudeImportance(p=2)

# 2. Initialize a pruner with the model and the importance criterion
ignored_layers = []
for m in model.modules():
    if isinstance(m, torch.nn.Linear) and m.out_features == 1000:
        ignored_layers.append(m) # DO NOT prune the final classifier!

pruner = tp.pruner.BasePruner( # We can always choose BasePruner if sparse training is not required.
    model,
    example_inputs,
    importance=imp,
    pruning_ratio=0.5, # remove 50% channels, ResNet18 = {64, 128, 256, 512} => ResNet18_Half = {32, 64, 128, 256}
    # pruning_ratio_dict = {model.conv1: 0.2, model.layer2: 0.8}, # customized pruning ratios for layers or blocks
    ignored_layers=ignored_layers,
    round_to=8, # It's recommended to round dims/channels to 4x or 8x for acceleration. Please see: https://docs.nvidia.com/deeplearning/performance/dl-performance-convolutional/index.html
)

# 3. Prune the model
base_macs, base_nparams = tp.utils.count_ops_and_params(model, example_inputs)
tp.utils.print_tool.before_pruning(model) # or print(model)
pruner.step()
tp.utils.print_tool.after_pruning(model) # or print(model), this util will show the difference before and after pruning
macs, nparams = tp.utils.count_ops_and_params(model, example_inputs)
print(f"MACs: {base_macs/1e9} G -> {macs/1e9} G, #Params: {base_nparams/1e6} M -> {nparams/1e6} M")


# 4. finetune the pruned model using your own code.
# finetune(model)
# ...

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False) => (conv1): Conv2d(3, 32, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True) => (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False) => (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True) => (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1

We can check that model layers were indeed changed

In [ ]:
model

ResNet(
  (conv1): Conv2d(3, 32, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

# Quantization

For quantization methods in PyTorch, see: https://pytorch.org/blog/introduction-to-quantization-on-pytorch/

## Automatic Mixed Precision (AMP) Training and Inference

AMP is easy to enable in PyTorch

In [ ]:
def traditional_training_loop(model, dataloader, optimizer, scheduler, criterion):
    model.train()
    for batch in dataloader:
        optimizer.zero_grad()
        output = model(**batch)
        batch.update(output)

        loss = criterion(**batch)
        loss.backward()
        optimizer.step()
        scheduler.step()

In [ ]:
def amp_training_loop(model, dataloader, optimizer, scheduler, criterion):
    model.train()

    # add scaler
    scaler = torch.amp.GradScaler("cuda")

    for batch in dataloader:
        optimizer.zero_grad()

        # Forward pass + loss under autocast
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            output = model(**batch)
            batch.update(output)
            loss = criterion(**batch)

        # Backward with scaled loss
        scaler.scale(loss).backward()

        # Optimizer step through scaler
        scaler.step(optimizer)
        # note: if there was an overflow, the step is skipped

        # Update scaling factor
        scaler.update()

        # even though the optimizer step was skipped, we do scheduler step
        # however, you may want to skip it too
        scheduler.step()

## HuggingFace Transformers

In [HuggingFace Transformers](https://huggingface.co/docs/transformers/v5.9.0/quantization/overview), several quantization back-ends are supported and can be applied to your model with as simple as just adding a few lines:

In [ ]:
!pip install -U transformers accelerate bitsandbytes

In [ ]:
def print_gpu_memory():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"Allocated: {allocated:.2f} GB")
    print(f"Reserved:  {reserved:.2f} GB")

In [ ]:
import time

def run_model(model):
    print("Memory before run")
    print_gpu_memory()
    prompt = "Explain quantization in one paragraph."

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    # 1 warm-up
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
        )

    start = time.perf_counter()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
        )
    total = time.perf_counter() - start

    print()
    print(tokenizer.decode(output[0], skip_special_tokens=True))
    print()
    print(f"Finished in {total} seconds")
    print("Memory after run")
    print_gpu_memory()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# 4-bit quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

model_4bit = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto",
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
run_model(model_4bit)

Memory before run
Allocated: 1.07 GB
Reserved:  1.12 GB


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Explain quantization in one paragraph.
assistant
Quantization is the process of converting continuous data into discrete values or symbols that represent it. It involves mapping an input range to a smaller output range, typically using a set of fixed-point numbers. This technique is widely used in various fields such as signal processing, image and video compression, machine learning, and artificial intelligence. Quantization reduces the precision of the data but can significantly decrease storage requirements and improve computational efficiency. The goal is to balance between accuracy loss and performance gains based on specific application needs.

Finished in 14.007125466999696 seconds
Memory after run
Allocated: 1.08 GB
Reserved:  1.18 GB


Restart session and try without 4-bit

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# FP16 baseline
model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
run_model(model_fp16)

Memory before run
Allocated: 2.88 GB
Reserved:  2.93 GB

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Explain quantization in one paragraph.
assistant
Quantization is the process of converting continuous-valued data into discrete values within a limited range or precision level. This technique reduces the amount of storage and transmission required for digital signals while maintaining their essential characteristics. By discretizing the signal's amplitude or intensity levels, quantization helps to minimize noise and improve the efficiency of communication systems, image processing, and other applications that require compact representations of information.

Finished in 6.330536263999875 seconds
Memory after run
Allocated: 2.88 GB
Reserved:  2.94 GB


While the memory benefit is clear, NF 4-bit is actually slower here. This is because we do activations in FP16 and use 4-bit precision only for storing weights. Dequantization adds overhead resulting in slower inference.